In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:

# --- CONFIGURATION ---
# Remplace par le chemin réel vers ton dossier contenant "WithMask" et "WithoutMask"
DATASET_PATH = "./" 


data = []

# Vérifie que le dossier existe
if not os.path.exists(DATASET_PATH):
    print(f"Attention: Le chemin {DATASET_PATH} n'existe pas.")
else:
    for class_name in os.listdir(DATASET_PATH):
        class_path = os.path.join(DATASET_PATH, class_name)
        
        if not os.path.isdir(class_path):
            continue
            
        print(f"Analyse de la classe : {class_name}...")
        print(len(os.listdir(class_path)))
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            
            try:
                # Ouverture de l'image et conversion en RGB (pour éviter les soucis avec PNG/RGBA ou N&B)
                with Image.open(img_path) as img:
                    img = img.convert('RGB')
                    width, height = img.size
                    resolution = f"{width}x{height}"
                    
                    # Conversion en tableau numpy pour les calculs de moyenne
                    img_array = np.array(img)
                    
                    # Moyenne par canal (0: R, 1: G, 2: B)
                    mean_r = img_array[:, :, 0].mean()
                    mean_g = img_array[:, :, 1].mean()
                    mean_b = img_array[:, :, 2].mean()
                    
                    data.append({
                        "class": class_name,
                        "resolution": resolution,
                        "mean_r": mean_r,
                        "mean_g": mean_g,
                        "mean_b": mean_b
                    })
            except Exception as e:
                # Ignore les fichiers qui ne sont pas des images (ex: .DS_Store)
                pass

# Création du DataFrame
df = pd.DataFrame(data)
print(f"\nTerminé ! {len(df)} images analysées au total.")
display(df.head())


Analyse de la classe : Test...
Analyse de la classe : Train...
Analyse de la classe : Validation...

Terminé ! 0 images analysées au total.


""


In [3]:


if not df.empty:
    top_10_shapes = df['resolution'].value_counts().head(10)
    
    plt.figure(figsize=(10, 6))
    bars = top_10_shapes.plot(kind='bar', color='#2f6fed', edgecolor='black')
    
    plt.title("Top 10 des résolutions d'images", fontsize=14, fontweight='bold')
    plt.xlabel("Résolution (Largeur x Hauteur)", fontsize=12)
    plt.ylabel("Nombre d'images", fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Ajout des valeurs au-dessus des barres
    for i, v in enumerate(top_10_shapes):
        plt.text(i, v + (v * 0.01), str(v), ha='center', va='bottom', fontweight='bold')
        
    plt.tight_layout()
    plt.show()

# %% [markdown]
# ## 3. Couleur moyenne par classe
# Équivalent du BarChart du dashboard. On regroupe par classe réelle et on affiche R, G et B.

# %%
if not df.empty:
    # Agrégation des moyennes par classe
    rgb_grouped = df.groupby('class')[['mean_r', 'mean_g', 'mean_b']].mean()
    
    classes = rgb_grouped.index
    x = np.arange(len(classes))
    width = 0.2  # Largeur des barres
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Création des 3 barres pour chaque classe (décalées sur l'axe X)
    ax.bar(x - width, rgb_grouped['mean_r'], width, label='Canal Rouge (R)', color='#8b0000', edgecolor='black')
    ax.bar(x,         rgb_grouped['mean_g'], width, label='Canal Vert (G)', color='#1b5e20', edgecolor='black')
    ax.bar(x + width, rgb_grouped['mean_b'], width, label='Canal Bleu (B)', color='#0d1a66', edgecolor='black')
    
    ax.set_title("Couleur moyenne des canaux RGB par classe", fontsize=14, fontweight='bold')
    ax.set_ylabel("Valeur moyenne du canal (0-255)", fontsize=12)
    ax.set_xlabel("Classe réelle", fontsize=12)
    
    ax.set_xticks(x)
    ax.set_xticklabels(classes)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()